exemple

In [ ]:
import folium
m = folium.Map(location=[37.6, 14.2],zoom_start=7)
m


In [ ]:
m = folium.Map(location=[37.6, 14.2],zoom_start=7)
folium.Marker(
    location=[37.76, 14.98],
    popup='ETNA'
    ).add_to(m)
m

Quêtes api geo coding 

Introduction
L'API Adresse est une API REST qui renvoie les coordonnées géographiques (latitude et longitude) à partir d'une adresse postale française. Si plusieurs adresses postales (pour cause d'imprécision ou de numéro de rue inexistant) correspondant à la requête sont trouvées, l'API retourne plusieurs coordonnées, avec à chaque fois un "score". Les coordonnées sont données dans l'ordre de score décroissant, vous pouvez donc retenir la première adresse.

In [ ]:
# Exécute le code suivant :
import requests
link = 'https://data.geopf.fr/geocodage/search/?q=728+Route+de+Villerest&postcode=42155'
r = requests.get(link).json()
r

In [6]:
# Les coordonnées sont données dans l'ordre longitude puis latitude
# Ici nous ne sélectionnons que la première adresse (indice 0)
print("première adresse :", r['features'][0])
print("coordonnées de la première adresse :",r['features'][0]['geometry']['coordinates'])
print("score de la première adresse :",r['features'][0]['properties']['score'])

première adresse : {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [3.993163, 46.00996]}, 'properties': {'label': '728 Route de Villerest 42155 Ouches', 'score': 0.9521245454545454, 'housenumber': '728', 'id': '42162_0008_00728', 'name': '728 Route de Villerest', 'postcode': '42155', 'citycode': '42162', 'x': 776847.88, 'y': 6546063.21, 'city': 'Ouches', 'context': '42, Loire, Auvergne-Rhône-Alpes', 'type': 'housenumber', 'importance': 0.47337, 'depcode': '42', 'street': 'Route de Villerest', '_type': 'address'}}
coordonnées de la première adresse : [3.993163, 46.00996]
score de la première adresse : 0.9521245454545454


In [7]:
# Pour plus de facilité, tu peux ajouter une limite sur le nombre d'éléments retournés

link = 'https://data.geopf.fr/geocodage/search/?q=728+Route+de+Villerest&postcode=42155'
r = requests.get(link).json()
print("SANS limite, cette adresse renvoie combien de coordonnées ?",len(r['features']))

link = 'https://data.geopf.fr/geocodage/search/?q=728+Route+de+Villerest&postcode=42155&limit=1'
r = requests.get(link).json()
print("AVEC limite, cette adresse renvoie combien de coordonnées ?",len(r['features']))

SANS limite, cette adresse renvoie combien de coordonnées ? 2
AVEC limite, cette adresse renvoie combien de coordonnées ? 1


Comment créer sa requête à l'API
A toi de modifier la chaine de caractère pour créer la bonne URL de requête

In [9]:
# On observe que la requête se compose d'une partie fixe, suivi de l'adresse à chercher
# Une URL ne peut pas comporter de caractère espace " ",
# et il faut éviter si possible d'avoir des caractères spéciaux ou des accents

link_main = 'https://data.geopf.fr/geocodage/search/?q='
adresse = '728 Route de Villerest, 42155 Ouches'


link = link_main + adresse.replace(" ", "%20")  # A toi de jouer ici

print(link)

https://data.geopf.fr/geocodage/search/?q=728%20Route%20de%20Villerest,%2042155%20Ouches


In [10]:
# Crée ici une fonction qui transforme une adresse postale en URL de requête pour l'API Adresse,
# puis effectue la requête et retourne les coordonnées :

import requests

def API_adresse(adresse_postale):
    url = "https://data.geopf.fr/geocodage/search/?q=" + adresse_postale.replace(" ", "%20")
    data = requests.get(url).json()
    return data["features"][0]["geometry"]["coordinates"]

In [13]:
# Teste-là ici :
API_adresse(adresse)

[3.993163, 46.00996]

DataViz
Les coordonnées peuvent être utilisées sur des outils de visualisation, qu'il s'agisse d'outils de BI (PowerBI, Tableau), ou des bibliothèques de DataViz en Python comme Plotly ou Folium.

Ici nous allons afficher une carte avec Folium.

In [ ]:
# Attention, le système latin retourne les coordonnées en (longitude, latitude),
# Alors que le système anglo-saxon est en (latitude, longitude), il faut donc inverser les coordonnées
# Ca tombe bien, vous savez le faire sur une liste avec un "pas" de moins un.
point = r['features'][0]['geometry']['coordinates'][::-1]
point

In [14]:
import requests

def API_adresse(adresse_postale):
    url = "https://data.geopf.fr/geocodage/search/?q=" + adresse_postale.replace(" ", "%20")
    return requests.get(url).json()['features'][0]['geometry']['coordinates'][::-1]

In [19]:
# La syntaxe de Folium est très simple, vous commencez par créer une carte centrée sur un point
# Vous pouvez modifier le niveau de zoom par défaut avec l'argument "zoom_start"
import requests, folium

point = requests.get("https://data.geopf.fr/geocodage/search/?q=728 Route de Villerest 42155 Ouches").json()['features'][0]['geometry']['coordinates'][::-1]

folium.Map(location=point, zoom_start=7)

In [18]:
# Puis vous pouvez ajouter des points de repère et mettre un commentaire cliquable
m = folium.Map(location=point, )
folium.Marker(
    location=point,
    popup='Un bon restau sur le pouce'
    ).add_to(m)
m